# Evaluasi Komprehensif Hybrid Ensemble & Ablation Study
**Proyek:** Analisis Sentimen Reputasi Undiksha pada Media Sosial  
**Tahap:** 6 – Ensemble, Uji Signifikansi Statistik, & Ablation Study (PRD §6 F7, F8, F9, F10)  


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import load_config
from src.evaluate.significance import mcnemar_test, run_pairwise_significance_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Tabel Perbandingan Model (Stratified 5-Fold Cross-Validation)

In [ ]:
comp_path = Path('../reports/tables/model_comparison.csv')
df_comp = pd.read_csv(comp_path)
df_comp


## 2. Visualisasi Perbandingan Metrik (Accuracy & F1-Macro)

In [ ]:
models = [m.replace("Komponen ", "").replace("Ensemble: ", "") for m in df_comp['Model']]
accs = [float(a.split('%')[0]) for a in df_comp['Accuracy']]
f1s = [float(f.split('%')[0]) for f in df_comp['F1-Macro']]

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(10, 5))
b1 = plt.bar(x - width/2, accs, width, label='Accuracy (%)', color='#3498db', edgecolor='black', alpha=0.85)
b2 = plt.bar(x + width/2, f1s, width, label='F1-Macro (%)', color='#2ecc71', edgecolor='black', alpha=0.85)

for b in b1:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{b.get_height():.1f}%", ha='center', fontsize=9)
for b in b2:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 1, f"{b.get_height():.1f}%", ha='center', fontsize=9, fontweight='bold')

plt.xticks(x, models, rotation=15, ha='right')
plt.ylabel('Persentase (%)')
plt.ylim(0, 105)
plt.title('Perbandingan Kinerja Komponen Tunggal vs Hybrid Ensemble (5-Fold CV)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 3. Hasil Ablation Study (Kontribusi Marginal Komponen)

In [ ]:
ablation_path = Path('../reports/tables/ablation_study.csv')
df_ablation = pd.read_csv(ablation_path)
df_ablation[['Konfigurasi Model', 'Accuracy', 'F1-Macro', 'Delta F1 vs Best Single']]


In [ ]:
configs = [c.split('. ')[-1].replace(" ★", "") for c in df_ablation['Konfigurasi Model']]
f1_ablation = [float(f.split('%')[0]) for f in df_ablation['F1-Macro']]

colors = ['#95a5a6', '#3498db', '#9b59b6', '#e67e22', '#1abc9c', '#f39c12', '#e74c3c', '#27ae60']

plt.figure(figsize=(10, 5))
bars = plt.barh(configs[::-1], f1_ablation[::-1], color=colors[::-1], edgecolor='black', alpha=0.9)
for b in bars:
    plt.text(b.get_width() + 1, b.get_y() + b.get_height()/2, f"{b.get_width():.2f}%", va='center', fontweight='bold')

plt.axvline(x=78.61, color='red', linestyle='--', label='Baseline Terbaik (SVM: 78.61%)')
plt.xlim(0, 100)
plt.xlabel('F1-Macro (%)')
plt.title('Ablation Study: Kontribusi Tiap Komponen pada Hybrid Ensemble')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()
